# Laboratorio 4: Análisis de Datos Geoespaciales
## Conexión y obtención de datos

Este notebook utiliza exclusivamente las fechas oficiales y descarga solo B03, B04 y B08, suficientes para calcular los índices NDVI y NDWI.

### 0. Dependencias

Las librerías utilizadas en este análisis están listadas en `requirements.txt`. La autenticación con Copernicus Data Space se realiza mediante OIDC.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Abra este notebook desde la carpeta raíz del proyecto DS-LAB04.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import pandas as pd
from configuracion import LAKES, OFFICIAL_SCENES, REQUIRED_BANDS
from descarga import (
    build_inventory, connect_copernicus, download_official_scene,
    save_inventory, update_download_status, validate_geotiff,
)

print(f'Bandas mínimas solicitadas: {REQUIRED_BANDS}')
print(f'Lagos configurados: {[item["display_name"] for item in LAKES.values()]}')

Bandas mínimas solicitadas: ('B03', 'B04', 'B08')
Lagos configurados: ['Lago de Amatitlán', 'Lago de Atitlán']


## 1. Conexión con la API Sentinel-2 (openEO)

La conexión a Copernicus Data Space permite consultar la colección Sentinel-2 L2A y generar las descargas necesarias para el análisis.

In [2]:
connection = connect_copernicus()
print('Conexión autenticada:', connection)

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=BQDA-QEFO 📋 to authenticate.

✅ Authorized successfully

Found orphaned SID: PySID:S-1-5-21-2957358142-2711924514-1296692830-3050305023
Found orphaned SID: PySID:S-1-5-21-3293864594-1002800012-2736569855-929768813
Authenticated using device code flow.
Conexión autenticada: <Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>


## 2. Inventario oficial de imágenes

El laboratorio exige trabajar únicamente con 11 fechas por lago. La validación siguiente debe dar exactamente 22 filas, 11 para Amatitlán y 11 para Atitlán.

In [3]:
inventory = build_inventory()
assert len(inventory) == 22, 'Deben existir 22 escenas oficiales en total.'
assert inventory.groupby('lago').size().to_dict() == {'amatitlan': 11, 'atitlan': 11}
inventory_path = save_inventory(inventory)
display(inventory)
print(f'Inventario guardado en: {inventory_path}')

,lago,nombre_lago,fecha,satelite,nubosidad_oficial_pct,bandas_solicitadas,ruta_salida,estado_descarga,observaciones
0,amatitlan,Lago de Amatitlán,2025-01-28,Sentinel-2B,0.06,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,
1,amatitlan,Lago de Amatitlán,2025-04-15,Sentinel-2A,0.09,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,
2,amatitlan,Lago de Amatitlán,2025-04-28,Sentinel-2B,1.03,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,
3,amatitlan,Lago de Amatitlán,2025-11-24,Sentinel-2B,0.50,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,
4,amatitlan,Lago de Amatitlán,2026-01-08,Sentinel-2C,0.77,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,
5,amatitlan,Lago de Amatitlán,2026-02-02,Sentinel-2B,0.39,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,
6,amatitlan,Lago de Amatitlán,2026-02-07,Sentinel-2C,0.02,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,Cobertura válida parcial aproximada: 57.1%.
7,amatitlan,Lago de Amatitlán,2026-03-29,Sentinel-2C,0.01,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,
8,amatitlan,Lago de Amatitlán,2026-04-13,Sentinel-2B,0.09,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,
9,amatitlan,Lago de Amatitlán,2026-04-28,Sentinel-2C,4.96,"B03,B04,B08",C:\Users\dijol\OneDrive\Documents\diego-drive\...,pendiente,


Inventario guardado en: C:\Users\dijol\OneDrive\Documents\diego-drive\Universidad\Ciclo 8\Data Science\DS-LAB04\outputs\tablas\inventario_datos.csv


In [4]:
# Control de trazabilidad: fechas, satélite y nubosidad reportados en la guía.
display(inventory.groupby('lago', as_index=False).agg(
    imagenes=('fecha', 'count'),
    primera_fecha=('fecha', 'min'),
    ultima_fecha=('fecha', 'max'),
    nubosidad_promedio_reportada=('nubosidad_oficial_pct', 'mean'),
))

,lago,imagenes,primera_fecha,ultima_fecha,nubosidad_promedio_reportada
0,amatitlan,11,2025-01-28,2026-06-19,1.901818
1,atitlan,11,2025-01-18,2026-07-22,2.456364


## 3. Descarga mínima de una escena

La descarga se limita a B03, B04 y B08. Se valida una imagen de cada lago antes de procesar el conjunto completo. La fecha 2026-02-07 de Amatitlán cuenta con cobertura válida parcial (aprox. 57.1%), observación que se mantiene en el inventario.

In [5]:
atitlan_test = download_official_scene(connection, 'atitlan', '2025-01-18')
amatitlan_test = download_official_scene(connection, 'amatitlan', '2025-01-28')
display(pd.DataFrame([validate_geotiff(atitlan_test), validate_geotiff(amatitlan_test)]))

0:00:00 Job 'j-2608132213224c81ba21d02094daa2cf': send 'start'
0:00:01 Job 'j-2608132213224c81ba21d02094daa2cf': queued (progress 0%)
0:00:07 Job 'j-2608132213224c81ba21d02094daa2cf': queued (progress 0%)
0:00:13 Job 'j-2608132213224c81ba21d02094daa2cf': queued (progress 0%)
0:00:22 Job 'j-2608132213224c81ba21d02094daa2cf': queued (progress 0%)
0:00:32 Job 'j-2608132213224c81ba21d02094daa2cf': queued (progress 0%)
0:00:44 Job 'j-2608132213224c81ba21d02094daa2cf': queued (progress 0%)
0:01:00 Job 'j-2608132213224c81ba21d02094daa2cf': running (progress N/A)
0:01:19 Job 'j-2608132213224c81ba21d02094daa2cf': running (progress N/A)
0:01:43 Job 'j-2608132213224c81ba21d02094daa2cf': running (progress N/A)
0:02:13 Job 'j-2608132213224c81ba21d02094daa2cf': finished (progress 100%)


c:\Users\dijol\OneDrive\Documents\diego-drive\Universidad\Ciclo 8\Data Science\DS-LAB04\src\descarga.py:108: UserDeprecationWarning: Call to deprecated method download_results. (Instead use `BatchJob.get_results` and the more flexible download functionality of `JobResults`) -- Deprecated since version 0.4.10.
  job.download_results(str(output_path))
C:\Users\dijol\AppData\Roaming\Python\Python314\site-packages\openeo\rest\job.py:202: UserDeprecationWarning: Call to deprecated method get_result. (Use `BatchJob.get_results` instead.) -- Deprecated since version 0.4.10.
  return self.get_result().download_files(target)
C:\Users\dijol\AppData\Roaming\Python\Python314\site-packages\openeo\rest\job.py:206: UserDeprecationWarning: Call to deprecated class _Result. (Use `JobResults` instead) -- Deprecated since version 0.4.10.
  return _Result(self)


0:00:00 Job 'j-2608132215444675a4f41f76bc7081a6': send 'start'
0:00:03 Job 'j-2608132215444675a4f41f76bc7081a6': queued (progress 0%)
0:00:08 Job 'j-2608132215444675a4f41f76bc7081a6': queued (progress 0%)
0:00:15 Job 'j-2608132215444675a4f41f76bc7081a6': queued (progress 0%)
0:00:23 Job 'j-2608132215444675a4f41f76bc7081a6': queued (progress 0%)
0:00:33 Job 'j-2608132215444675a4f41f76bc7081a6': running (progress N/A)
0:00:45 Job 'j-2608132215444675a4f41f76bc7081a6': running (progress N/A)
0:01:01 Job 'j-2608132215444675a4f41f76bc7081a6': running (progress N/A)
0:01:21 Job 'j-2608132215444675a4f41f76bc7081a6': running (progress N/A)
0:01:45 Job 'j-2608132215444675a4f41f76bc7081a6': finished (progress 100%)


,ruta,bandas,crs,ancho,alto,resolucion,nodata
0,C:\Users\dijol\OneDrive\Documents\diego-drive\...,3,EPSG:32615,2759,1751,"(10.0, 10.0)",-32768.0
1,C:\Users\dijol\OneDrive\Documents\diego-drive\...,3,EPSG:32615,1360,917,"(10.0, 10.0)",-32768.0


In [ ]:

for row in inventory.itertuples(index=False):
    path = download_official_scene(connection, row.lago, row.fecha)
    print(f'Descargado: {path}')

Descargado: C:\Users\dijol\OneDrive\Documents\diego-drive\Universidad\Ciclo 8\Data Science\DS-LAB04\data\raw\amatitlan\amatitlan_2025-01-28_B03_B04_B08.tif
0:00:00 Job 'j-2608132218144afabed2f5c93056e857': send 'start'
0:00:01 Job 'j-2608132218144afabed2f5c93056e857': created (progress 0%)
0:00:06 Job 'j-2608132218144afabed2f5c93056e857': queued (progress 0%)
0:00:13 Job 'j-2608132218144afabed2f5c93056e857': queued (progress 0%)
0:00:21 Job 'j-2608132218144afabed2f5c93056e857': queued (progress 0%)
0:00:31 Job 'j-2608132218144afabed2f5c93056e857': queued (progress 0%)
0:00:44 Job 'j-2608132218144afabed2f5c93056e857': queued (progress 0%)
0:00:59 Job 'j-2608132218144afabed2f5c93056e857': finished (progress 100%)


c:\Users\dijol\OneDrive\Documents\diego-drive\Universidad\Ciclo 8\Data Science\DS-LAB04\src\descarga.py:108: UserDeprecationWarning: Call to deprecated method download_results. (Instead use `BatchJob.get_results` and the more flexible download functionality of `JobResults`) -- Deprecated since version 0.4.10.
  job.download_results(str(output_path))
C:\Users\dijol\AppData\Roaming\Python\Python314\site-packages\openeo\rest\job.py:202: UserDeprecationWarning: Call to deprecated method get_result. (Use `BatchJob.get_results` instead.) -- Deprecated since version 0.4.10.
  return self.get_result().download_files(target)
C:\Users\dijol\AppData\Roaming\Python\Python314\site-packages\openeo\rest\job.py:206: UserDeprecationWarning: Call to deprecated class _Result. (Use `JobResults` instead) -- Deprecated since version 0.4.10.
  return _Result(self)


Descargado: C:\Users\dijol\OneDrive\Documents\diego-drive\Universidad\Ciclo 8\Data Science\DS-LAB04\data\raw\amatitlan\amatitlan_2025-04-15_B03_B04_B08.tif
0:00:00 Job 'j-2608132219184b299ad58e39786ef8bf': send 'start'
0:00:02 Job 'j-2608132219184b299ad58e39786ef8bf': queued (progress 0%)
0:00:07 Job 'j-2608132219184b299ad58e39786ef8bf': queued (progress 0%)
0:00:14 Job 'j-2608132219184b299ad58e39786ef8bf': queued (progress 0%)
0:00:22 Job 'j-2608132219184b299ad58e39786ef8bf': queued (progress 0%)
0:00:32 Job 'j-2608132219184b299ad58e39786ef8bf': queued (progress 0%)
0:00:44 Job 'j-2608132219184b299ad58e39786ef8bf': queued (progress 0%)
0:01:00 Job 'j-2608132219184b299ad58e39786ef8bf': running (progress N/A)
0:01:19 Job 'j-2608132219184b299ad58e39786ef8bf': running (progress N/A)
0:01:43 Job 'j-2608132219184b299ad58e39786ef8bf': running (progress N/A)
0:02:14 Job 'j-2608132219184b299ad58e39786ef8bf': finished (progress 100%)
Descargado: C:\Users\dijol\OneDrive\Documents\diego-drive\Uni

## 4. Control de calidad de las descargas

El inventario registra el estado `descargado_validado` cuando el archivo es georreferenciado y contiene las tres bandas solicitadas.

In [ ]:
inventory_final = update_download_status(inventory)
save_inventory(inventory_final)
display(inventory_final[['lago', 'fecha', 'satelite', 'bandas_solicitadas', 'estado_descarga', 'observaciones']])

pendientes = inventory_final.query("estado_descarga != 'descargado_validado'")
print(f'Escenas validadas: {(inventory_final.estado_descarga == "descargado_validado").sum()} de {len(inventory_final)}')
if not pendientes.empty:
    print('Pendientes o con incidencias:')
    display(pendientes[['lago', 'fecha', 'estado_descarga', 'observaciones']])

## Consideraciones para el análisis

El archivo `outputs/tablas/inventario_datos.csv` conserva la trazabilidad de cada imagen y los GeoTIFF validados contienen B03, B04 y B08. La delimitación final del análisis se realiza con el GeoJSON o una máscara precisa del lago, evitando incluir píxeles de tierra en las métricas. Las incidencias de cobertura o nubes quedan registradas durante la validación.